In [1]:
from arcs.generate import GraphGenerator
from importlib import resources

In [175]:
gg = GraphGenerator()
tabledict = gg.from_file(
    filename=resources.files("arcs").joinpath("data/quantum_data.json.gz"),
    temperature=248,
    pressure=20,
    max_reaction_length=5,
    graph=False
)

/Users/badw/github-projects/arcs/src/arcs/generate.py:537: UserWarning: NOHSO4: 1 imag modes removed
  warnings.warn(f"{compound}: {warn_info['warning']}", UserWarning)


In [176]:
import pandas as pd 
df = pd.DataFrame(tabledict)

In [177]:
from collections import defaultdict 
def table_refactor(tabledict):
    refactored_dict = defaultdict(dict)
    for i, _dict in tabledict.items():
        gibbs_free_energy = _dict["G"]
        if gibbs_free_energy < 0: 
            reactants = _dict["reactants"]
            products = _dict["products"]
            refactored_dict[i]["K"] = _dict["K"]
            refactored_dict[i]["G"] = _dict["G"]
            for compound,num in reactants.items():
                refactored_dict[i][compound] = -num # -ve as being removed 
            for compound,num in products.items():
                refactored_dict[i][compound] = num # +ve as being made
        else:
            reactants = _dict["products"]
            products = _dict["reactants"]
            refactored_dict[i]["K"] = _dict["K_rev"]
            refactored_dict[i]["G"] = _dict["G_rev"]
            for compound,num in reactants.items():
                refactored_dict[i][compound] = -num # -ve as being removed 
            for compound,num in products.items():
                refactored_dict[i][compound] = num # +ve as being made
    return refactored_dict 

refactor = table_refactor(tabledict)

In [178]:
refactored_df = pd.DataFrame(refactor)

In [179]:
import numpy as np 

In [219]:
# 1. get weighted random_compounds 
concs = {'H2O':50,'SO2':1,'NO2':20,'NO':13,"O2":73,"N2O4":1}
# convert to probabilities 
p_1 = {k: v / sum(concs.values()) for k, v in concs.items()}
p_1

{'H2O': 0.31645569620253167,
 'SO2': 0.006329113924050633,
 'NO2': 0.12658227848101267,
 'NO': 0.08227848101265822,
 'O2': 0.4620253164556962,
 'N2O4': 0.006329113924050633}

In [220]:
# 2. weighted shuffle random 
#keys, weights = list(p_1.keys()), list(p_1.values())
#order = np.random.choice(keys, size=len(keys), replace=False, p=weights)
#reordered = {k: p_1[k] for k in order}#

#weighted_shuffle(p_1)

### based on claude help 

1. find based on availability 
2. based on Gibbs 
3. based on extra availability i.e. 50% O2 means reactions with more stoichiometric O2 will be better 

In [221]:
import numpy as np
import pandas as pd

conc = p_1
df = refactored_df.T

species = [c for c in df.columns if c not in ["G","K"]]
avail   = np.array([conc.get(s, 0.0) for s in species])
present = avail > 0

S     = df[species].to_numpy(float)
gibbs = df['G'].to_numpy(float)

reac = np.clip(-S, 0, None)    # reactants: negative coeffs -> positive amounts
prod = np.clip( S, 0, None)    # products:  positive coeffs

# --- Criterion 4: EVERY reactant must be present ---
reac_mask = reac > 0
have_all  = ((reac_mask & ~present).sum(axis=1) == 0) & reac_mask.any(axis=1)

# --- Criteria 2 & 3: reactant availability, weighted by stoichiometry ---
availability = reac @ avail

def minmax(x):
    lo, hi = np.nanmin(x), np.nanmax(x)
    return np.zeros_like(x) if hi == lo else (x - lo) / (hi - lo)

gibbs_term = 1 - minmax(gibbs)   # most negative -> ~1
avail_term = minmax(availability)
w_gibbs, w_avail = 0.3, 0.7
score = w_gibbs * gibbs_term + w_avail * avail_term

def side(a):
    return " + ".join(f"{v:g} {s}" for s, v in zip(species, a) if v > 0)
eqn = [f"{side(reac[i])} = {side(prod[i])}" for i in range(len(df))]

out = df.assign(reaction=eqn, availability=availability, score=score)
ranked = out.loc[have_all].sort_values('score', ascending=False)

# --- guardrail: no reaction with a missing reactant may survive ---
assert ((reac[have_all] > 0) & ~present).sum() == 0, "infeasible reaction leaked into ranked!"

In [225]:
ranked.head(20).sort_values(by="G")

,K,G,O3,O2,CH2O,CH3COOH,N2O2,NO,N2O4,NO2,...,CH3NH2,H2SO3,C2H5NH2,NOHSO4,CS2,CH3_C6H5,CH3_CO_CH3,reaction,availability,score
469,9.892174e+89,-4.428535,NaN,-3.0,NaN,NaN,NaN,-4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3 O2 + 4 NO + 2 H2O = 4 HNO3,NaN,NaN
445,1.894439e+50,-2.474080,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 O2 + 2 H2O + 2 SO2 = 2 H2SO4,NaN,NaN
623,1.555380e+40,-1.977781,NaN,-1.0,NaN,NaN,NaN,NaN,-2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 O2 + 2 N2O4 + 2 H2O = 4 HNO3,NaN,NaN
168,3.165225e+35,-1.746922,NaN,NaN,NaN,NaN,NaN,-4.0,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4 NO = 2 NO2 + 1 N2,NaN,NaN
542,1.437556e+33,-1.631637,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,-4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 O2 + 4 NO2 + 2 H2O = 4 HNO3,NaN,NaN
98,9.622738e+31,-1.573851,NaN,NaN,NaN,NaN,NaN,-4.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4 NO = 1 N2O4 + 1 N2,NaN,NaN
913,5.634096e+28,-1.414785,NaN,NaN,NaN,NaN,NaN,NaN,-4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4 N2O4 + 3 H2O = 1 N2O + 6 HNO3,NaN,NaN
28,2.623213e+28,-1.398449,NaN,-1.0,NaN,NaN,NaN,-2.0,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 O2 + 2 NO = 2 NO2,NaN,NaN
5,7.974944e+24,-1.225377,NaN,-1.0,NaN,NaN,NaN,-2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 O2 + 2 NO = 1 N2O4,NaN,NaN
358,6.971168e+24,-1.222502,NaN,-1.0,NaN,NaN,NaN,-4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 O2 + 4 NO + 2 H2O = 4 HNO2,NaN,NaN


In [212]:
np.random.choice(list(ranked.head(20).T))

np.int64(3)

### Todo 

1. figure out next stage of algorithm

    a) continue as normal ARCS algorithm:

        - random weighted choice of reaction
        - proceed down a random path 


    b) new algorithm:

        - randomly sample i.e. 1k times the ranked reactions here 
        - update concentrations 
        - generate a new ranked table, and sample that 1k times 
        - update concentrations 
        - do this N times until "converged" (if it does converge?)


    c) alternative new algorithm 
    
        - run all the top 20 ranked reactions 
        - update concentrations 
        - do it again 
        - etc. 
        - (make sure to randomly order the reactions each time)